<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/experiments/mean_escape_time/double_well/double_well_mean_escape_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Double-well mean escape time: five-method comparison

This is the principal double-well notebook for the paper. It places the original five experiments in one sequence, with shared Monte Carlo labels, one reference solution, consistent progress output, resumable Adam checkpoints, and side-by-side evaluation.

`quick` checks the workflow with smaller counts and the same shallow `tanh` network used by the committor notebook. `paper` restores the recorded double-well architecture and numerical settings and is intended for a long Colab run.


## 1. Stochastic system and exit-time PDE

Let

$$\Phi(x,y)=A\left(1-e^{-((x-1)^2+(y-1/2)^2)/\nu^2}\right)
\left(1-e^{-((x+1)^2+(y+1/2)^2)/\nu^2}\right),$$

$b=-\nabla\Phi$, and

$$dX_t=b(X_t)\,dt+\begin{pmatrix}\sqrt{2\epsilon}&0\\0&\sqrt{\epsilon}\end{pmatrix}dB_t.$$

The safe domain is the ellipse $\Omega=\{(x,y):(x/a)^2+(y/b)^2<1\}$ with $a=2$ and $b=1.5$. The mean exit time solves

$$-\mathcal L\tau=1\quad\text{in }\Omega,\qquad \tau=0\quad\text{on }\partial\Omega,$$

$$\mathcal L\tau=\epsilon\tau_{xx}+\frac{\epsilon}{2}\tau_{yy}+b_1\tau_x+b_2\tau_y.$$

The recorded parameters are $A=5$, $\nu=0.1$, and $\epsilon=1.2$.


## 2. Compared methods

The methods and adaptive objective match the OU notebook. The default `quick` run uses two width-64 `tanh` hidden layers, as in the committor notebook; the `paper` preset restores the recorded three width-30 hidden layers and initialization. Section 6 exposes an ordinary `nn.Module` and a single network factory, so readers may replace the supplied MLP with any scalar PyTorch architecture without changing the boundary ansatz or losses. The standard PINN provides the sampled-boundary baseline; the variational PINN provides a weak-form alternative; and the three exact-boundary powers show how the boundary factor affects the approximation.

| Method | Trial function | Active loss components |
|---|---|---|
| Standard | $N_\theta$ | strong PDE, boundary, Monte Carlo data |
| Boundary-adapted | $dN_\theta$ | strong PDE, Monte Carlo data |
| Over-enforced | $d^{3/2}N_\theta$ | strong PDE, Monte Carlo data |
| Under-enforced | $d^{1/2}N_\theta$ | strong PDE, Monte Carlo data |
| Variational/Nitsche | $N_\theta$ | weak residual, Nitsche boundary, Monte Carlo data |

The strong residual is $\mathcal Lu_\theta+1$. The standard PINN enforces $u=0$ with sampled boundary values; the three powers of the ellipse distance factor enforce it in the trial function. The Monte Carlo loss compares $u_\theta(z_j)$ with mean exit-time labels $\widehat\tau_j$.

If $J_i$ are the active components in the table, the original adaptive objective is

$$J_{\rm train}=\sum_i e^{-2s_i}J_i+\sum_i s_i,$$

with trainable log scales $s_i$. Checkpoint selection now uses this same weighted objective. In the original standard double-well notebook, the selection score omitted the boundary and data terms. The correction changes which epoch is saved as best; it does not change the training objective.


In [ ]:
# Pure-Python Colab setup. scikit-fem replaces the old system-level FEniCS install.
import importlib.util
import math
import subprocess
import sys
import time
from collections import OrderedDict
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if importlib.util.find_spec("skfem") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-fem>=11,<13"])

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import pandas as pd
import torch
from torch import nn
from skfem import Basis, BilinearForm, ElementTriP1, LinearForm, MeshTri, asm, condense, solve

torch.set_default_dtype(torch.float64)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", DEVICE)


## 3. Editable configuration

The finite-element implementation changes from FEniCS to `scikit-fem`, but it assembles the same P1 weak problem. Consequently, the historical metrics below remain FEniCS-era outputs and must not be attributed to this cell until the experiment is rerun.

The `quick` preset uses the public default shallow network and reduces sample counts, Monte Carlo resolution and horizon, mesh refinement, and iteration budgets. It leaves the equation, boundary ansatzes, and loss definitions unchanged. In Colab, the setup mounts Google Drive when `SAVE_TO_DRIVE` is true so labels and checkpoints survive a disconnected runtime.


In [ ]:
RUN_SCALE = "quick"                 # "quick" or "paper"
METHODS = ["standard", "adapted", "over", "under", "variational"]
RUN_REFERENCE = True
RUN_TRAINING = False
LOAD_SAVED_MODELS = False
RUN_SMOKE_CHECK = True
RUN_LBFGS = False
RESUME = True
SAVE_TO_DRIVE = True
SEED = 2026

potential_height = 5.0
well_width = 0.1
diffusion_epsilon = 1.2
ellipse_a, ellipse_b = 2.0, 1.5

if RUN_SCALE == "paper":
    width, depth = 30, 3
    initialization_gain = 2.0
    pool_size, batch_size, adam_epochs = 8192, 512, 5_000
    data_anchors, paths_per_anchor = 256, 256
    mc_dt, mc_max_time = 1e-3, 20.0
    fem_refinements = 7
    lbfgs_iterations = 3 * adam_epochs
    report_every, checkpoint_every = 100, 500
elif RUN_SCALE == "quick":
    width, depth = 64, 2
    initialization_gain = None
    pool_size, batch_size, adam_epochs = 384, 96, 30
    data_anchors, paths_per_anchor = 20, 20
    mc_dt, mc_max_time = 2e-3, 2.0
    fem_refinements = 4
    lbfgs_iterations = 20
    report_every, checkpoint_every = 10, 10
else:
    raise ValueError("RUN_SCALE must be 'quick' or 'paper'")

learning_rate = 5e-4
number_of_decays = 100
final_learning_rate_factor = 1e-3
if drive is not None and SAVE_TO_DRIVE:
    drive.mount("/content/drive", force_remount=False)
    output_root = Path("/content/drive/MyDrive/boundary-adapted-pinns-runs/double_well")
else:
    output_root = Path("runs/double_well")
output_root.mkdir(parents=True, exist_ok=True)

experiment_settings = {
    "run_scale": RUN_SCALE, "seed": SEED, "potential_height": potential_height,
    "well_width": well_width, "diffusion_epsilon": diffusion_epsilon,
    "ellipse_a": ellipse_a, "ellipse_b": ellipse_b,
    "width": width, "depth": depth, "initialization_gain": initialization_gain,
    "pool_size": pool_size, "batch_size": batch_size, "adam_epochs": adam_epochs,
    "data_anchors": data_anchors, "paths_per_anchor": paths_per_anchor,
    "mc_dt": mc_dt, "mc_max_time": mc_max_time,
    "learning_rate": learning_rate, "number_of_decays": number_of_decays,
    "final_learning_rate_factor": final_learning_rate_factor,
}

def require_matching_settings(record, expected, saved_path):
    if record.get("settings") != expected:
        raise ValueError(
            f"{saved_path} was created with different settings. "
            "Set RESUME = False to create new files, or restore the saved settings."
        )

torch.manual_seed(SEED)
np.random.seed(SEED)
print({"scale": RUN_SCALE, "output": str(output_root)})


## 4. Drift and finite-element reference

The weak reference problem is

$$\int_\Omega\left(\epsilon u_xv_x+\frac{\epsilon}{2}u_yv_y-(b\cdot\nabla u)v\right)dx
=\int_\Omega v\,dx,$$

with homogeneous Dirichlet data. The paper runs used FEniCS on a generated ellipse mesh. The cell below uses continuous piecewise-linear elements in `scikit-fem`, making the Colab setup straightforward while keeping the equation and weak form fixed.


In [ ]:
def drift_numpy(x, y):
    d1 = (x - 1.0)**2 + (y - 0.5)**2
    d2 = (x + 1.0)**2 + (y + 0.5)**2
    e1, e2 = np.exp(-d1 / well_width**2), np.exp(-d2 / well_width**2)
    f1, f2 = 1.0 - e1, 1.0 - e2
    dphi_x = potential_height * ((2 * (x - 1.0) / well_width**2) * e1 * f2 + f1 * (2 * (x + 1.0) / well_width**2) * e2)
    dphi_y = potential_height * ((2 * (y - 0.5) / well_width**2) * e1 * f2 + f1 * (2 * (y + 0.5) / well_width**2) * e2)
    return -dphi_x, -dphi_y


def solve_fem_reference():
    mesh = MeshTri.init_circle(nrefs=fem_refinements).scaled((ellipse_a, ellipse_b))
    basis = Basis(mesh, ElementTriP1())

    @BilinearForm
    def operator(u, v, w):
        bx, by = drift_numpy(w.x[0], w.x[1])
        return (diffusion_epsilon * u.grad[0] * v.grad[0]
                + 0.5 * diffusion_epsilon * u.grad[1] * v.grad[1]
                - (bx * u.grad[0] + by * u.grad[1]) * v)

    @LinearForm
    def forcing(v, w):
        return v

    matrix = asm(operator, basis)
    rhs = asm(forcing, basis)
    coefficients = solve(*condense(matrix, rhs, D=basis.get_dofs()))
    return mesh, basis, coefficients


fem_mesh = fem_basis = fem_coefficients = None
if RUN_REFERENCE:
    fem_mesh, fem_basis, fem_coefficients = solve_fem_reference()
    print(f"FEM reference: {fem_basis.N} P1 degrees of freedom")


### A sample exit trajectory

This Euler–Maruyama path uses the same drift, anisotropic noise, ellipse, and exit rule as the Monte Carlo labels. The simulator accepts the starting point and numerical settings explicitly, so it can also be used to make explanatory trajectory figures without the training code.


In [ ]:
def simulate_double_well_trajectory(
    start=(0.0, 0.0), *, dt=1e-3, max_time=20.0, seed=SEED + 1,
):
    rng = np.random.default_rng(seed)
    point = np.asarray(start, dtype=float).copy()
    path = [point.copy()]
    exited = (point[0] / ellipse_a)**2 + (point[1] / ellipse_b)**2 >= 1.0
    noise = np.array([math.sqrt(2 * diffusion_epsilon * dt), math.sqrt(diffusion_epsilon * dt)])
    for _ in range(int(max_time / dt)):
        if exited:
            break
        bx, by = drift_numpy(point[0], point[1])
        point = point + np.array([bx, by]) * dt + noise * rng.standard_normal(2)
        path.append(point.copy())
        exited = (point[0] / ellipse_a)**2 + (point[1] / ellipse_b)**2 >= 1.0
    return np.asarray(path), (len(path) - 1) * dt, exited


trajectory, trajectory_time, trajectory_exited = simulate_double_well_trajectory()
angle = np.linspace(0.0, 2.0 * np.pi, 300)
fig, axis = plt.subplots(figsize=(6, 5))
axis.plot(trajectory[:, 0], trajectory[:, 1], lw=1.0)
axis.plot(ellipse_a * np.cos(angle), ellipse_b * np.sin(angle), "k--", label="exit boundary")
axis.scatter(*trajectory[0], label="start")
axis.scatter(*trajectory[-1], marker="x", label="exit" if trajectory_exited else "time cap")
axis.set(aspect="equal", title=f"Double-well path: {'exit' if trajectory_exited else 'censored'} at t={trajectory_time:.3g}")
axis.legend()
plt.show()


## 5. Shared sampling and Monte Carlo data

Interior points are uniform by area in the ellipse. Boundary angles are uniform, matching the original notebooks. Boundary losses and Nitsche integrals multiply each sample by the ellipse's local arclength factor, avoiding the angular bias of an unweighted ellipse parameterization. For each anchor, Euler–Maruyama advances

$$X_{n+1}=X_n+b(X_n)\Delta t+
\begin{pmatrix}\sqrt{2\epsilon\Delta t}&0\\0&\sqrt{\epsilon\Delta t}\end{pmatrix}\xi_n,$$

and the label is the mean simulated exit time. Paths that have not exited by the time cap are recorded at the cap, with the per-anchor censored fractions printed and saved.

The original one-method notebooks generated their Monte Carlo samples separately. This notebook generates one sample and reuses it for all five methods, changing the random realization but not the simulation rule, sample counts, or training losses.


In [ ]:
def sample_ellipse(count, *, device=DEVICE):
    angle = 2 * math.pi * torch.rand(count, device=device)
    radius = torch.sqrt(torch.rand(count, device=device))
    return torch.stack((ellipse_a * radius * torch.cos(angle), ellipse_b * radius * torch.sin(angle)), dim=1)


def sample_boundary(count, *, device=DEVICE):
    angle = 2 * math.pi * torch.rand(count, device=device)
    return torch.stack((ellipse_a * torch.cos(angle), ellipse_b * torch.sin(angle)), dim=1)


def drift_torch(points):
    x, y = points[..., 0], points[..., 1]
    d1 = (x - 1.0).square() + (y - 0.5).square()
    d2 = (x + 1.0).square() + (y + 0.5).square()
    e1, e2 = torch.exp(-d1 / well_width**2), torch.exp(-d2 / well_width**2)
    f1, f2 = 1.0 - e1, 1.0 - e2
    dphi_x = potential_height * ((2 * (x - 1.0) / well_width**2) * e1 * f2 + f1 * (2 * (x + 1.0) / well_width**2) * e2)
    dphi_y = potential_height * ((2 * (y - 0.5) / well_width**2) * e1 * f2 + f1 * (2 * (y + 0.5) / well_width**2) * e2)
    return torch.stack((-dphi_x, -dphi_y), dim=-1)


@torch.no_grad()
def monte_carlo_exit_time(x0, n_paths, dt, max_time):
    x0 = x0.to(DEVICE)
    batch = x0.shape[0]
    positions = x0[:, None, :].expand(batch, n_paths, 2).clone()
    elapsed = torch.zeros(batch, n_paths, device=DEVICE)
    alive = torch.ones(batch, n_paths, dtype=torch.bool, device=DEVICE)
    max_steps = int(max_time / dt)
    interval = max(1, max_steps // 100)
    noise_scale = torch.tensor([math.sqrt(2 * diffusion_epsilon * dt), math.sqrt(diffusion_epsilon * dt)], device=DEVICE)
    started = time.time()
    for step in range(max_steps):
        if not alive.any():
            break
        current = positions[alive]
        positions[alive] = current + drift_torch(current) * dt + torch.randn_like(current) * noise_scale
        elapsed[alive] += dt
        active_indices = alive.nonzero(as_tuple=False)
        active = positions[alive]
        exited = (active[:, 0] / ellipse_a).square() + (active[:, 1] / ellipse_b).square() >= 1.0
        if exited.any():
            ids = active_indices[exited]
            alive[ids[:, 0], ids[:, 1]] = False
        if step % interval == 0 or step + 1 == max_steps:
            print(f"MC step {step + 1:6d}/{max_steps} | alive {alive.float().mean():6.2%} | {time.time() - started:6.1f}s", flush=True)
    if alive.any():
        print(f"Censored at {max_time:g}: {int(alive.sum())}/{alive.numel()} paths")
    return elapsed.mean(dim=1), alive.float().mean(dim=1)


## 6. Models and strong residual

The normalized ellipse boundary factor is

$$d(x)=\frac{\omega(x)}{\sqrt{\omega(x)^2+|\nabla\omega(x)|^2}},\qquad
\omega(x)=1-(x/a)^2-(y/b)^2.$$

Powers $1$, $3/2$, and $1/2$ give the three exact-boundary trial spaces.

`CoordinateNetwork` is only the supplied default. Edit it, or replace `make_base_network()`, with any scalar PyTorch architecture. The method wrappers below remain responsible for the boundary factor and adaptive weights.


In [ ]:
METHOD_SPECS = OrderedDict([
    ("standard", {"power": None, "losses": ("pde", "data", "bc")}),
    ("adapted", {"power": 1.0, "losses": ("pde", "data")}),
    ("over", {"power": 1.5, "losses": ("pde", "data")}),
    ("under", {"power": 0.5, "losses": ("pde", "data")}),
    ("variational", {"power": None, "losses": ("var", "data", "bc")}),
])


def smooth_distance(points):
    x, y = points[:, 0], points[:, 1]
    omega = 1.0 - (x / ellipse_a).square() - (y / ellipse_b).square()
    grad_squared = (2 * x / ellipse_a**2).square() + (2 * y / ellipse_b**2).square()
    return omega / torch.sqrt(omega.square() + grad_squared)


class CoordinateNetwork(nn.Module):
    """Supplied default; replace this class or make_base_network() freely."""

    def __init__(self):
        super().__init__()
        layers = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers.extend((nn.Linear(width, width), nn.Tanh()))
        layers.append(nn.Linear(width, 1))
        self.layers = nn.Sequential(*layers)
        if initialization_gain is not None:
            self.apply(self._initialize)

    @staticmethod
    def _initialize(module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_normal_(module.weight, gain=initialization_gain)
            nn.init.constant_(module.bias, initialization_gain)

    def forward(self, points):
        return self.layers(points)


def make_base_network():
    """Return any nn.Module with scalar output shape [n] or [n, 1]."""
    return CoordinateNetwork()


class DoubleWellModel(nn.Module):
    def __init__(self, method):
        super().__init__()
        self.method = method
        self.power = METHOD_SPECS[method]["power"]
        self.network = make_base_network()
        self.log_scales = nn.ParameterDict({name: nn.Parameter(torch.tensor(-0.5)) for name in METHOD_SPECS[method]["losses"]})

    def forward(self, points):
        raw = self.network(points)
        if raw.ndim == 1:
            raw = raw.unsqueeze(1)
        if raw.shape != (points.shape[0], 1):
            raise ValueError("The base network must return shape [n] or [n, 1].")
        if self.power is None:
            return raw
        return raw * smooth_distance(points).clamp_min(0).pow(self.power).unsqueeze(1)


def strong_residual(model, points):
    x = points.detach().clone().requires_grad_(True)
    tau = model(x).squeeze(1)
    grad = torch.autograd.grad(tau.sum(), x, create_graph=True)[0]
    tau_xx = torch.autograd.grad(grad[:, 0].sum(), x, create_graph=True)[0][:, 0]
    tau_yy = torch.autograd.grad(grad[:, 1].sum(), x, create_graph=True)[0][:, 1]
    drift = drift_torch(x)
    generator = diffusion_epsilon * tau_xx + 0.5 * diffusion_epsilon * tau_yy + (drift * grad).sum(dim=1)
    return generator + 1.0


### Variational/Nitsche loss

With $D=\operatorname{diag}(\epsilon,\epsilon/2)$, integration by parts gives

$$
\int_\Omega \nabla u\cdot D\nabla\phi\,dx
-\int_\Omega(b\cdot\nabla u)\phi\,dx
-\int_{\partial\Omega}(D\nabla u\cdot n)\phi\,ds
=\int_\Omega\phi\,dx.
$$

The symmetric Nitsche term is $-\int_{\partial\Omega}u(D\nabla\phi\cdot n)\,ds$, and $\int_{\partial\Omega}u\phi\,ds$ supplies the separately weighted boundary component. Fifty Gaussian tests are used with length scale $|\Omega|/10$. Volume integrals are estimated by uniform-area sample averages. Boundary angles are sampled uniformly and multiplied by $|x'(\theta)|$, so the boundary terms approximate arclength integrals rather than angular averages.


In [ ]:
n_test_functions = 50
test_length = math.pi * ellipse_a * ellipse_b / 10.0
test_centers = sample_ellipse(n_test_functions)
def boundary_arclength_factor(points):
    """Return |d(a cos(theta), b sin(theta))/d theta| at boundary points."""
    x, y = points[:, 0], points[:, 1]
    return torch.sqrt((ellipse_a * y / ellipse_b).square() + (ellipse_b * x / ellipse_a).square())


def variational_residual(model, interior, boundary):
    interior = interior.detach().clone().requires_grad_(True)
    boundary = boundary.detach().clone().requires_grad_(True)
    u_int = model(interior).squeeze(1)
    grad_u = torch.autograd.grad(u_int.sum(), interior, create_graph=True)[0]
    delta = interior[:, None, :] - test_centers[None, :, :]
    phi = torch.exp(-delta.square().sum(dim=2) / (2 * test_length**2))
    grad_phi = -delta * phi[..., None] / test_length**2
    area = math.pi * ellipse_a * ellipse_b
    drift = drift_torch(interior.detach())
    diffusion_term = (diffusion_epsilon * grad_u[:, None, 0] * grad_phi[:, :, 0]
                      + 0.5 * diffusion_epsilon * grad_u[:, None, 1] * grad_phi[:, :, 1])
    drift_term = -(drift * grad_u).sum(dim=1, keepdim=True) * phi
    weak = (diffusion_term + drift_term).mean(dim=0) * area - phi.mean(dim=0) * area

    delta_b = boundary[:, None, :] - test_centers[None, :, :]
    phi_b = torch.exp(-delta_b.square().sum(dim=2) / (2 * test_length**2))
    grad_phi_b = -delta_b * phi_b[..., None] / test_length**2
    normal_raw = torch.stack((boundary[:, 0] / ellipse_a**2, boundary[:, 1] / ellipse_b**2), dim=1)
    normal = normal_raw / torch.linalg.norm(normal_raw, dim=1, keepdim=True)
    u_b = model(boundary).squeeze(1)
    grad_u_b = torch.autograd.grad(u_b.sum(), boundary, create_graph=True)[0]
    flux_u = diffusion_epsilon * grad_u_b[:, 0] * normal[:, 0] + 0.5 * diffusion_epsilon * grad_u_b[:, 1] * normal[:, 1]
    ds_dtheta = boundary_arclength_factor(boundary)
    consistency = -2 * math.pi * (flux_u[:, None] * phi_b * ds_dtheta[:, None]).mean(dim=0)
    flux_phi = (diffusion_epsilon * grad_phi_b[:, :, 0] * normal[:, None, 0]
                + 0.5 * diffusion_epsilon * grad_phi_b[:, :, 1] * normal[:, None, 1])

    symmetry = -2 * math.pi * (u_b[:, None] * flux_phi * ds_dtheta[:, None]).mean(dim=0)
    raw_penalty = 2 * math.pi * (u_b[:, None] * phi_b * ds_dtheta[:, None]).mean(dim=0)
    return weak + consistency + symmetry, raw_penalty


def adaptive_objective(model, components):
    return sum(torch.exp(-2 * model.log_scales[name]) * value + model.log_scales[name] for name, value in components.items())


def component_losses(model, method, interior, boundary, data_x, data_y):
    data_loss = (model(data_x).squeeze(1) - data_y).square().mean()
    if method == "variational":
        weak, penalty = variational_residual(model, interior, boundary)
        return {"var": weak.square().mean(), "data": data_loss, "bc": penalty.square().mean()}
    losses = {"pde": strong_residual(model, interior).square().mean(), "data": data_loss}
    if method == "standard":
        values = model(boundary).squeeze(1).square()
        weights = boundary_arclength_factor(boundary)
        losses["bc"] = (values * weights).sum() / weights.sum()
    return losses


In [ ]:
def run_model_smoke_check():
    interior = sample_ellipse(8)
    boundary = sample_boundary(8)
    data_x = sample_ellipse(6)
    data_y = torch.zeros(6, device=DEVICE)
    for method in METHOD_SPECS:
        model = DoubleWellModel(method).to(DEVICE)
        components = component_losses(model, method, interior, boundary, data_x, data_y)
        objective = adaptive_objective(model, components)
        if not torch.isfinite(objective):
            raise RuntimeError(f"{method} smoke check produced a non-finite objective")
        objective.backward()
    print("Model smoke check passed for all five methods.")


if RUN_SMOKE_CHECK:
    run_model_smoke_check()


## 7. Prepare the shared labels

The Monte Carlo anchors and labels are generated once and reused by all methods. They are saved with their simulation settings for resumption.


In [ ]:
data_path = output_root / f"double_well_{RUN_SCALE}_shared_data.pt"
data_settings = {
    "format_version": 2,
    "seed": SEED, "potential_height": potential_height, "well_width": well_width,
    "diffusion_epsilon": diffusion_epsilon, "ellipse_a": ellipse_a, "ellipse_b": ellipse_b,
    "data_anchors": data_anchors, "paths_per_anchor": paths_per_anchor,
    "mc_dt": mc_dt, "mc_max_time": mc_max_time,
    "evaluation_interior_count": pool_size,
    "evaluation_boundary_count": max(1, pool_size // 4),
}

if data_path.exists() and RESUME:
    shared = torch.load(data_path, map_location=DEVICE)
    require_matching_settings(shared, data_settings, data_path)
    x_data, tau_data = shared["x"].to(DEVICE), shared["tau"].to(DEVICE)
    data_censored = shared["censored"].to(DEVICE)
    evaluation_interior = shared["evaluation_interior"].to(DEVICE)
    evaluation_boundary = shared["evaluation_boundary"].to(DEVICE)
    print("Loaded shared labels and evaluation points:", data_path)
else:
    x_data = sample_ellipse(data_anchors)
    tau_data, data_censored = monte_carlo_exit_time(x_data, paths_per_anchor, mc_dt, mc_max_time)
    evaluation_interior = sample_ellipse(pool_size)
    evaluation_boundary = sample_boundary(max(1, pool_size // 4))
    torch.save({
        "x": x_data.cpu(), "tau": tau_data.cpu(), "censored": data_censored.cpu(),
        "evaluation_interior": evaluation_interior.cpu(),
        "evaluation_boundary": evaluation_boundary.cpu(),
        "settings": data_settings,
    }, data_path)
    print("Saved shared labels and evaluation points:", data_path)

print(f"Shared labels: {len(x_data)} anchors × {paths_per_anchor} paths")
print(f"Censored fraction: mean={float(data_censored.mean()):.2%}, max={float(data_censored.max()):.2%}")


## 8. Training and checkpoints

The paper schedule uses 5,000 Adam epochs, exponential learning-rate decay, then up to 15,000 L-BFGS iterations. `best_adam.pt` now means the smallest full weighted objective for that method. `last.pt` is a separate resumable state, so a periodic save no longer overwrites the best checkpoint.

Set `LOAD_SAVED_MODELS = True` and keep the same settings to evaluate completed `final.pt` files in a fresh runtime without retraining.


In [ ]:
def make_scheduler(optimizer):
    step = max(1, adam_epochs // number_of_decays)
    gamma = math.exp(math.log(final_learning_rate_factor) / number_of_decays)
    return torch.optim.lr_scheduler.StepLR(optimizer, step_size=step, gamma=gamma)


def detached_row(components):
    return {name: float(value.detach()) for name, value in components.items()}


def train_method(method):
    run_dir = output_root / f"{RUN_SCALE}_{method}"
    run_dir.mkdir(parents=True, exist_ok=True)
    model = DoubleWellModel(method).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = make_scheduler(optimizer)
    history, start_epoch, best_score = [], 0, math.inf
    last_path, best_path = run_dir / "last.pt", run_dir / "best_adam.pt"
    checkpoint_settings = {**experiment_settings, "method": method}
    if RESUME and last_path.exists():
        state = torch.load(last_path, map_location=DEVICE)
        require_matching_settings(state, checkpoint_settings, last_path)
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        history, start_epoch, best_score = state["history"], state["epoch"], state["best_score"]
        print(f"{method}: resuming after epoch {start_epoch}")

    started = time.time()
    for epoch in range(start_epoch + 1, adam_epochs + 1):
        model.train()
        optimizer.zero_grad()
        components = component_losses(model, method, sample_ellipse(batch_size), sample_boundary(batch_size), x_data, tau_data)
        objective = adaptive_objective(model, components)
        objective.backward()
        optimizer.step()
        scheduler.step()
        model.eval()
        validation = component_losses(model, method, evaluation_interior, evaluation_boundary, x_data, tau_data)
        score = float(adaptive_objective(model, validation).detach())
        row = {"epoch": epoch, "objective": float(objective.detach()), "checkpoint_objective": score, **detached_row(components)}
        history.append(row)
        if score < best_score:
            best_score = score
            torch.save({"model": model.state_dict(), "epoch": epoch, "score": score, "settings": checkpoint_settings}, best_path)
        if epoch % report_every == 0 or epoch == 1 or epoch == adam_epochs:
            pieces = " | ".join(f"{key} {value:.3e}" for key, value in detached_row(components).items())
            print(f"{method:11s} epoch {epoch:5d}/{adam_epochs} | {pieces} | {time.time() - started:.1f}s", flush=True)
        if epoch % checkpoint_every == 0 or epoch == adam_epochs:
            torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "history": history, "epoch": epoch, "best_score": best_score, "settings": checkpoint_settings}, last_path)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE)["model"])
    if RUN_LBFGS:
        fixed_interior, fixed_boundary = sample_ellipse(pool_size), sample_boundary(max(1, pool_size // 4))
        lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=lbfgs_iterations, history_size=500, tolerance_grad=0, tolerance_change=0, line_search_fn="strong_wolfe")
        calls = 0

        def closure():
            nonlocal calls
            calls += 1
            lbfgs.zero_grad()
            parts = component_losses(model, method, fixed_interior, fixed_boundary, x_data, tau_data)
            loss = adaptive_objective(model, parts)
            loss.backward()
            if calls % 100 == 0:
                print(method, "L-BFGS call", calls, detached_row(parts), flush=True)
            return loss

        lbfgs.step(closure)
    torch.save({"model": model.state_dict(), "history": history, "method": method, "scale": RUN_SCALE, "settings": {**checkpoint_settings, "lbfgs": RUN_LBFGS, "lbfgs_iterations": lbfgs_iterations}}, run_dir / "final.pt")
    return model, pd.DataFrame(history)


In [ ]:
trained = {}
histories = {}
if RUN_TRAINING:
    for method in METHODS:
        if method not in METHOD_SPECS:
            raise ValueError(f"Unknown method: {method}")
        trained[method], histories[method] = train_method(method)
    print("Training complete:", ", ".join(trained))
elif LOAD_SAVED_MODELS:
    for method in METHODS:
        run_dir = output_root / f"{RUN_SCALE}_{method}"
        final_path = run_dir / "final.pt"
        state = torch.load(final_path, map_location=DEVICE)
        expected = {**experiment_settings, "method": method, "lbfgs": RUN_LBFGS, "lbfgs_iterations": lbfgs_iterations}
        require_matching_settings(state, expected, final_path)
        model = DoubleWellModel(method).to(DEVICE)
        model.load_state_dict(state["model"])
        trained[method] = model
        histories[method] = pd.DataFrame(state["history"])
    print("Loaded final models:", ", ".join(trained))
else:
    print("Set RUN_TRAINING = True to train, or LOAD_SAVED_MODELS = True to evaluate saved final models.")


## 9. Evaluation against the FEM reference

The grid-based $L^1$, $L^2$, $H^1$, and $H^2$ calculations follow the historical notebooks. A new run uses the `scikit-fem` solution created above; the retained paper table in Section 10 used the old FEniCS reference.


In [ ]:
grid_n = 200
gx, gy = np.meshgrid(np.linspace(-ellipse_a, ellipse_a, grid_n), np.linspace(-ellipse_b, ellipse_b, grid_n))
if fem_mesh is not None:
    triangulation = mtri.Triangulation(fem_mesh.p[0], fem_mesh.p[1], fem_mesh.t.T)
    ellipse_mask = triangulation.get_trifinder()(gx, gy) >= 0
else:
    ellipse_mask = (gx / ellipse_a)**2 + (gy / ellipse_b)**2 < 1.0
grid_points = np.column_stack((gx[ellipse_mask], gy[ellipse_mask]))


def fem_values(points):
    if fem_basis is None:
        raise RuntimeError("Run the finite-element reference cell first.")
    return np.asarray(fem_basis.probes(np.asarray(points).T) @ fem_coefficients)


def evaluate_model(model):
    model.eval()
    with torch.no_grad():
        prediction = model(torch.as_tensor(grid_points, device=DEVICE)).squeeze(1).cpu().numpy()
    truth = fem_values(grid_points)
    predicted_grid, reference_grid = np.zeros_like(gx), np.zeros_like(gx)
    predicted_grid[ellipse_mask], reference_grid[ellipse_mask] = prediction, truth
    dx, dy = 2 * ellipse_a / (grid_n - 1), 2 * ellipse_b / (grid_n - 1)
    error = predicted_grid - reference_grid
    error_y, error_x = np.gradient(error, dy, dx)
    error_xx = np.gradient(error_x, dx, axis=1)
    error_yy = np.gradient(error_y, dy, axis=0)
    error_xy = np.gradient(error_y, dx, axis=1)
    truth_y, truth_x = np.gradient(reference_grid, dy, dx)
    truth_xx = np.gradient(truth_x, dx, axis=1)
    truth_yy = np.gradient(truth_y, dy, axis=0)
    truth_xy = np.gradient(truth_y, dx, axis=1)
    rel_l1 = np.mean(np.abs(prediction - truth)) / np.mean(np.abs(truth))
    rel_l2 = np.sqrt(np.mean((prediction - truth)**2) / np.mean(truth**2))
    rel_h1 = np.sqrt(np.mean(error[ellipse_mask]**2 + error_x[ellipse_mask]**2 + error_y[ellipse_mask]**2) / np.mean(reference_grid[ellipse_mask]**2 + truth_x[ellipse_mask]**2 + truth_y[ellipse_mask]**2))
    rel_h2 = np.sqrt(np.mean(error[ellipse_mask]**2 + error_x[ellipse_mask]**2 + error_y[ellipse_mask]**2 + error_xx[ellipse_mask]**2 + error_yy[ellipse_mask]**2 + 2 * error_xy[ellipse_mask]**2) / np.mean(reference_grid[ellipse_mask]**2 + truth_x[ellipse_mask]**2 + truth_y[ellipse_mask]**2 + truth_xx[ellipse_mask]**2 + truth_yy[ellipse_mask]**2 + 2 * truth_xy[ellipse_mask]**2))
    return predicted_grid, reference_grid, {"L1 relative": rel_l1, "L2 relative": rel_l2, "H1 relative": rel_h1, "H2 relative": rel_h2}


if trained and fem_basis is not None:
    fields, metrics, reference_grid = {}, {}, None
    for name, model in trained.items():
        fields[name], reference_grid, metrics[name] = evaluate_model(model)
    display(pd.DataFrame(metrics).T)
    figures = {"FEM reference": reference_grid, **fields}
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    for axis, (name, values) in zip(axes.flat, figures.items()):
        plot = axis.contourf(gx, gy, values, levels=50)
        axis.set_title(name)
        axis.set_aspect("equal")
        fig.colorbar(plot, ax=axis)
    for axis in axes.flat[len(figures):]:
        axis.axis("off")
    plt.show()


## 10. Historical paper-run summary

These aggregate values were retained as output in the original result notebook. The underlying `dw.txt` log is absent. They used the FEniCS reference and were produced before the corrected checkpoint selection, corrected array shapes in the variational data and boundary losses, and arclength weighting of ellipse boundary integrals. They are therefore not presented as a rerun of this notebook.

| Method | Relative L² | Relative H¹ | Relative H² |
|---|---:|---:|---:|
| Boundary-adapted | 2.643e-1 ± 3.512e-4 | 3.233e-1 ± 8.386e-4 | 4.031e-1 ± 1.300e-3 |
| Standard | 2.953e-1 ± 1.200e-2 | 4.499e-1 ± 1.393e-2 | 6.530e-1 ± 2.366e-2 |
| Variational/Nitsche | 5.639e-1 ± 1.100e-2 | 1.173 ± 9.637e-2 | 1.197e1 ± 1.703 |
| Over-enforced $d^{3/2}$ | 4.857e-1 ± 3.981e-1 | 5.865e-1 ± 4.187e-1 | 8.943e-1 ± 2.767e-1 |
| Under-enforced $d^{1/2}$ | 9.047e-1 ± 8.524e-3 | 9.202e-1 ± 9.625e-3 | 9.625e-1 ± 8.305e-3 |

The retained training-loss summary from the same runs is:

| Method | PDE or weak loss | Data loss | Boundary loss |
|---|---:|---:|---:|
| Boundary-adapted | 2.316e-4 ± 7.213e-6 | 2.734e-3 ± 2.190e-4 | — |
| Standard | 3.580 ± 5.859e-1 | 9.612e-3 ± 2.935e-3 | 1.466e-11 ± 8.607e-12 |
| Variational/Nitsche | 1.059e-6 ± 6.009e-7 | 1.050e-1 ± 9.377e-3 | 8.653e-18 ± 9.463e-18 |
| Over-enforced $d^{3/2}$ | 6.912 ± 5.865 | 1.565e-1 ± 2.674e-1 | — |
| Under-enforced $d^{1/2}$ | 1.088 ± 2.784e-2 | 4.127e-1 ± 1.489e-2 | — |


## Adapting the method to another PDE

Keep the training and reporting flow, and replace the problem-specific pieces: `drift_numpy`/`drift_torch`, the diffusion coefficients in `strong_residual`, the domain samplers, the boundary factor, and the reference solver. For a parameterized boundary, include its surface-Jacobian weight in boundary integrals as this ellipse example does. Derive $-\mathcal L u=1$ first to check every sign. `make_base_network()` accepts any scalar PyTorch architecture independently of these numerical choices.


## References

- N. Tepakbong, J. Fan, X. Zhou, and D.-X. Zhou, [“Boundary-Adapted PINNs for Elliptic Dirichlet Problems”](https://arxiv.org/abs/2607.19167), 2026.
- T. Gustafsson and G. D. McBain, [“scikit-fem: A Python package for finite element assembly”](https://doi.org/10.21105/joss.02369), *JOSS* 5(52), 2369 (2020).
- M. Raissi, P. Perdikaris, and G. E. Karniadakis, [“Physics-informed neural networks”](https://doi.org/10.1016/j.jcp.2018.10.045), *JCP* 378 (2019).
- N. Sukumar and A. Srivastava, [“Exact imposition of boundary conditions with distance functions”](https://doi.org/10.1016/j.cma.2021.114333), *CMAME* 389 (2022).
- Y. Liao and P. Ming, [“Deep Nitsche Method”](https://doi.org/10.4208/cicp.OA-2020-0219), *Commun. Comput. Phys.* 29 (2021).
- A. Kendall, Y. Gal, and R. Cipolla, [“Multi-task learning using uncertainty to weigh losses”](https://openaccess.thecvf.com/content_cvpr_2018/html/Kendall_Multi-Task_Learning_Using_CVPR_2018_paper.html), CVPR 2018.
